- In the `lensing_transform` function, removed `dL` from input parameters and `delta_phi` from output parameters. `M_lz` and `dL` are highly degenerate (their Jacobian entries are $\sim 20$ orders of magnitude smaller than other nontrivial entries); I decided to keep `M_lz` because it's specific to the AGN-CBC model. Then we also need to remove an output parameter because the current code supports square Jacobians only; I picked `delta_phi` arbitrarily.
- This doesn't solve our problem, which is that the Fisher matrices themselves are nearly singular
- For testing purposes I put everything in float128, but it's overkill for non-nearly-singular Jacobians

## Module imports

In [121]:
import mpmath
mpmath.mp.dps = 35  # ~float128 precision (18-19 sig. digits) with margin

from collections import OrderedDict
from pathlib import Path
from functools import partial
from multiprocessing import Pool, cpu_count
from typing import Union

import jax.numpy as np
import numpy as onp
import matplotlib.pyplot as plt

from gwfast.gwfastGlobals import detectors as det_dict, detPath, MTSUN_SI, DAY_TO_SEC
import gwfast.waveforms as waveforms
from gwfast.detector import Detector
import gwfast.network as network
from gwfast.signals import AGNLensedGWSignal, GeneralLensedGWSignal
from gwfast.fisherTools import reduce_Fisher_matrix, compute_covariance_matrix

## Function definitions

### `high_dim_matmul`

In [78]:
def high_dim_matmul(mat_1, mat_2):
    """
    Perform matrix multiplication for high-dimensional arrays.
    The first two dimensions of the input arrays are treated as matrices.
    """
    return onp.einsum('ij...,jk...->ik...', mat_1, mat_2)


### `covariance_change_variable`

In [ ]:
# ── mpmath helpers ─────────────────────────────────────────────────────────

def _to_mp(arr):
    """Convert a 2-D float128 numpy array to an mpmath matrix (full precision)."""
    arr = onp.ascontiguousarray(onp.asarray(arr, dtype=onp.float128))
    n, m = arr.shape
    M = mpmath.matrix(n, m)
    for i in range(n):
        for j in range(m):
            M[i, j] = mpmath.mpf(format(float(arr[i, j]), '.35e'))
    return M

def mp_cond(arr):
    """Condition number via mpmath SVD (σ_max / σ_min)."""
    M = _to_mp(arr)
    S = mpmath.svd(M, compute_uv=False)
    s = sorted([abs(v) for v in S])
    if s[0] == 0:
        return float('inf')
    return float(s[-1] / s[0])

def mp_slogdet_sym(arr):
    """
    (sign, log|det|) for a symmetric positive-(semi)-definite matrix via
    mpmath Cholesky: det = prod(diag(L))^2, always sign >= 0.
    Falls back to LU if Cholesky fails.
    """
    M = _to_mp(arr)
    try:
        L = mpmath.cholesky(M)
        log_abs_det = float(2 * sum(mpmath.log(abs(L[i, i])) for i in range(M.rows)))
        return 1, log_abs_det
    except Exception:
        # Fall back to LU for indefinite/singular matrices
        return _mp_slogdet_lu(M)

def _mp_slogdet_lu(M):
    """(sign, log|det|) via mpmath LU decomposition."""
    try:
        P, L, U = mpmath.lu(M)
        # det = det(P) * prod(diag(U))
        # det(P): count row swaps in permutation matrix P
        P_arr = onp.array([[float(P[i, j]) for j in range(P.cols)] for i in range(P.rows)])
        # permutation sign: number of row swaps
        perm_sign = int(round(onp.linalg.det(P_arr)))
        log_abs_det = float(sum(mpmath.log(abs(U[i, i])) for i in range(U.rows)))
        diag_signs = [int(mpmath.sign(U[i, i])) for i in range(U.rows)]
        det_sign = perm_sign * int(onp.prod(diag_signs))
        return det_sign, log_abs_det
    except Exception:
        return 0, float('-inf')

def mp_slogdet_subblock(arr_subblock):
    """
    (sign, log|det|) for the small sub-Jacobian block using LU.
    More reliable than eigenvalue method for a small 5×5 matrix.
    """
    M = _to_mp(arr_subblock)
    return _mp_slogdet_lu(M)


# ── float128 Jacobian via central finite differences ───────────────────────

def float128_jacobian(f, x0, eps=None):
    """
    Jacobian via central finite differences, fully in float128 precision.
    f  : callable (1-D float128 array) -> (1-D float128 array)
    x0 : 1-D float128 array
    Returns J of shape (m_out, n_params) in float128.
    """
    x0 = onp.asarray(x0, dtype=onp.float128)
    n = len(x0)
    f0 = onp.asarray(f(x0), dtype=onp.float128)
    m_out = len(f0)
    if eps is None:
        eps = onp.float128(1e-10)
    J = onp.zeros((m_out, n), dtype=onp.float128)
    for i in range(n):
        x_p = x0.copy(); x_p[i] += eps
        x_m = x0.copy(); x_m[i] -= eps
        fp = onp.asarray(f(x_p), dtype=onp.float128)
        fm = onp.asarray(f(x_m), dtype=onp.float128)
        J[:, i] = (fp - fm) / (2 * eps)
    return J


def _print_matrix_table(mat, row_labels, col_labels, fmt='.3g', col_w=None):
    """Print a 2-D matrix as a table with row and column labels."""
    mat = onp.asarray(mat, dtype=float)
    if col_w is None:
        col_w = max(max(len(k) for k in col_labels), 10) + 2
    lbl_w = max(len(k) for k in row_labels) + 2
    header = ' ' * lbl_w + ''.join(f'{k:>{col_w}}' for k in col_labels)
    print(header)
    print('-' * len(header))
    for i, row_key in enumerate(row_labels):
        row_str = f'{row_key:>{lbl_w}}' + ''.join(
            f'{mat[i, j]:{col_w}{fmt}}' for j in range(mat.shape[1])
        )
        print(row_str)


# ── covariance_change_variable ──────────────────────────────────────────────

def covariance_change_variable(
        convariance_matrix, injection_parameters, transform, from_params
    ):
    onp.set_printoptions(precision=3)
    full_rank = convariance_matrix.shape[0]
    param_shape = convariance_matrix.shape[2:]
    matrix_keys = list(injection_parameters.keys())
    keys_indices = [matrix_keys.index(key) for key in from_params]

    sub_injection_parameters = OrderedDict(
        {key: onp.atleast_1d(injection_parameters[key]).reshape(-1) for key in from_params}
    )

    sub_transformed_parameters = transform(sub_injection_parameters)
    out_keys = list(sub_transformed_parameters.keys())

    def _flat_transform(x_flat):
        """for a single matrix"""
        x_flat = onp.asarray(x_flat, dtype=onp.float128)
        params = OrderedDict(
            {k: onp.atleast_1d(x_flat[i]) for i, k in enumerate(from_params)}
        )
        result = transform(params)
        return onp.array(
            [float(onp.squeeze(result[k])) for k in out_keys],
            dtype=onp.float128
        )

    if convariance_matrix.ndim == 2:
        x0 = onp.array(
            [float(sub_injection_parameters[k][0]) for k in from_params],
            dtype=onp.float128
        )

        # Sanity check
        f0 = _flat_transform(x0)
        if not onp.all(onp.isfinite(f0)):
            raise ValueError(f"Transform returns non-finite values at x0: {f0}")

        # Jacobian in float128 via central finite differences
        jac_mat = float128_jacobian(_flat_transform, x0)  # (Nout, Nin), nxn sub-block

        if not onp.all(onp.isfinite(jac_mat)):
            print(f"Warning: Jacobian contains non-finite values:\n{jac_mat}")
            transform_covar = onp.full((full_rank, full_rank), onp.nan, dtype=onp.float128)
            transform_keys = list(matrix_keys)
            for idx, new_key_name in zip(keys_indices, out_keys):
                transform_keys[idx] = new_key_name
            transform_parameters = {
                key: onp.asarray(injection_parameters.get(key, sub_transformed_parameters.get(key)))
                for key in transform_keys
            }
            return transform_covar, transform_parameters, transform_keys

        full_jacobian_mat = onp.eye(full_rank, dtype=onp.float128)
        full_jacobian_mat[onp.ix_(keys_indices, keys_indices)] = jac_mat

        J = full_jacobian_mat
        cov_f128 = convariance_matrix.astype(onp.float128)
        transform_covar = J @ cov_f128 @ J.T

        print("== Condition number (Jacobian sub-block) ==", mp_cond(jac_mat))

        # Build parameter labels (rows = output params, cols = input params)
        diag_labels = list(matrix_keys)
        for idx, new_key in zip(keys_indices, out_keys):
            diag_labels[idx] = new_key

        print("\n== Full Jacobian (rows=output params, cols=input params) ==")
        _print_matrix_table(J, diag_labels, matrix_keys)

        # det(J) = det(jac_mat) since J = I with sub-block replaced
        sign_C, logdet_C = mp_slogdet_sym(cov_f128)
        print('== log|det(C)| (original covariance) ==', sign_C, logdet_C)

        sign_J, logdet_J = mp_slogdet_subblock(jac_mat)
        print('== log|det(J)| (sub-Jacobian, = det of full J) ==', sign_J, logdet_J)
        print('== Expected log|det(J C J^T)| = 2*logdetJ + logdetC ==',
              sign_J**2 * sign_C, 2 * logdet_J + logdet_C)

        sign_T, logdet_T = mp_slogdet_sym(transform_covar)
        print('== log|det(J C J^T)| (transformed covariance) ==', sign_T, logdet_T)

    # Maintain original order of keys, replacing transformed subset
    transform_keys = list(matrix_keys)
    for idx, new_key_name in zip(keys_indices, out_keys):
        transform_keys[idx] = new_key_name

    transform_parameters = {}
    for key in transform_keys:
        value = injection_parameters.get(key, None)
        if value is None:
            value = sub_transformed_parameters.get(key, None)
        transform_parameters[key] = onp.asarray(value)

    return transform_covar, transform_parameters, transform_keys


### Lensing transform utilities

In [153]:

from astropy.cosmology import Planck18 as cosmo
from gwfast.gwfastGlobals import MRSUN_SI, MTSUN_SI, uGpc, DAY_TO_SEC
# from gwfast.old_lensing_utils import _get_alpha_hat

zGridGlob = onp.logspace(start=-6, stop=5, base=10, num=7000)
dLGridGlob = cosmo.luminosity_distance(zGridGlob).value / 1000.0  # Gpc

def _get_alpha_hat(R_orbit, approx=1):
    np = onp
    """
    Compute deflection angle from the orbital radius
    between the BBH and the SMBH.

    This assumes β = 0.

    R_orbit -- Unit: Schwarschild radius
    """
    approx_simp = np.sqrt(2 / R_orbit)
    match approx:
        ## Approx 1: The simplest approximation
        ## assuming α(x) to the first order
        case 1:
            return approx_simp
        ## Approx 2: Fit with log(r) vs log(err)
        ## still assuming α(x) to the first order
        case 2:
            idx = -0.5415779752686682
            y0 = -0.6327303836364937
            return (1 + 10 ** (y0) * R_orbit ** (idx)) * approx_simp
        ## Approx 3: Fit with log(r) vs log(err)
        ## assuming α(x) to the second order
        case 3:
            idx = -0.5042733754506686
            y0 = -0.2727560615461613
            return (1 + 10 ** (y0) * R_orbit ** (idx)) * approx_simp

def lensing_transform(lensing_parameters):
    """
    4 to 4 transform.

    Inputs (from_params): iota, R_orbit, src_pos, M_lz
    Outputs: delta_iota, relative_mass, relative_distance, delta_time
    """
    np = onp
    lensing_parameters = OrderedDict(lensing_parameters)
    # Use fiducial values for phase, psi, and dL, which doesn't affect Jacobian results
    lensing_parameters['phase'] = np.zeros_like(lensing_parameters['iota']) + 0.1
    lensing_parameters['psi']   = np.zeros_like(lensing_parameters['iota']) + 0.1
    lensing_parameters['dL'] = np.ones_like(lensing_parameters['iota'])
    outputs = compute_lensed_angles_approx(lensing_parameters)

    phenom_changes = {}
    phenom_changes['delta_iota']    = outputs['iota_m']  - outputs['iota_p']
    # phenom_changes['delta_phi']     = outputs['phase_m'] - outputs['phase_p']
    phenom_changes['relative_mass'] = (1 + outputs['z_rel_m']) / (1 + outputs['z_rel_p'])
    relative_magification = outputs['sqrt_mu_p'] / outputs['sqrt_mu_m']
    phenom_changes['relative_distance'] = \
        relative_magification * ((1 + outputs['z_rel_m']) / (1 + outputs['z_rel_p'])) ** 2
    phenom_changes['delta_time']    = outputs['delta_time']
    return phenom_changes

def compute_lensed_angles_approx(
        agn_bbh_system_params, angular_distances=False):
    np = onp
    parameters = agn_bbh_system_params.copy()
    iota = parameters["iota"]
    phase = parameters["phase"]
    psi = parameters["psi"]
    r_orbit = parameters["R_orbit"]  # R_Sch
    y_src = parameters["src_pos"]  # R_orbit

    # Useful constructs
    phi_N = np.pi / 2 - phase

    theta_E, beta, lens_mass_src = get_agn_lens_angles(
        parameters["M_lz"], r_orbit, y_src, parameters["dL"],
        angular_distances=angular_distances)

    # Image positions
    img_pos_1, img_pos_2 = PML_image_position(beta, theta_E)  # Radian

    # The opening angles
    alpha_hat = _get_alpha_hat(r_orbit)  # rad
    theta_bar_p = alpha_hat - (img_pos_1 - beta)
    theta_bar_m = alpha_hat - (img_pos_2 + beta)

    # Setting phi_N = 0 gives - delta_phi
    delta_phi = - get_phi_L(iota, y_src, 0)

    inv_Delta = (np.cos(iota)**2 + np.sin(iota)**2 * np.sin(delta_phi)**2)**-0.5
    iota_term = np.cos(iota) * np.cos(delta_phi) * inv_Delta
    phi_term = np.sin(delta_phi) / np.sin(iota) * inv_Delta
    psi_term = np.sin(iota) * np.sqrt(np.tan(iota)**2 + 1 / np.sin(delta_phi)**2)
    speed_term = np.sin(iota) * np.cos(delta_phi) * inv_Delta

    iota_p = iota - theta_bar_p * iota_term
    iota_m = iota + theta_bar_m * iota_term
    phi_p = phi_N + theta_bar_p * phi_term
    phi_m = phi_N - theta_bar_m * phi_term
    psi_p = psi + theta_bar_p / psi_term
    psi_m = psi + theta_bar_m / psi_term

    v_orb = Keplerian_speed(r_orbit)
    gamma = Lorentz_factor(v_orb)
    v_proj = - v_orb * np.sin(iota) * np.sin(delta_phi)
    v_orb_p = v_proj * (1 + theta_bar_p * speed_term)
    v_orb_m = v_proj * (1 - theta_bar_m * speed_term)

    z_rel_p = gamma * (1 + v_orb_p) - 1
    z_rel_m = gamma * (1 + v_orb_m) - 1
    z_grav = gravitational_redshift(r_orbit)

    delta_time, mu_p, mu_m = PML_time_delay_magnification(beta, theta_E)
    delta_time *= lens_mass_src * MTSUN_SI  # s
    sqrt_mu_p = np.sqrt(np.abs(mu_p))
    sqrt_mu_m = np.sqrt(np.abs(mu_m))

    return {
        'iota_p': iota_p,
        'iota_m': iota_m,
        'phase_p': np.pi/2 - phi_p,
        'phase_m': np.pi/2 - phi_m,
        'psi_p': psi_p,
        'psi_m': psi_m,
        'v_proj_p': v_orb_p,
        'v_proj_m': v_orb_m,
        'z_rel_p': z_rel_p,
        'z_rel_m': z_rel_m,
        'z_grav': z_grav,
        'delta_time': delta_time,
        'sqrt_mu_p': sqrt_mu_p,
        'sqrt_mu_m': sqrt_mu_m,
    }

def get_agn_lens_angles(redshifted_lens_mass, r_orbit, source_position,
                        luminosity_distance, angular_distances=False):
    np = onp
    '''
    Computes the Einstein angle, the source position angle, and the lens mass.

    Parameters
    ----------
    redshifted_lens_mass: float / array-like
        The redshifted lens mass in the detector frame, in solar masses.
    r_orbit: float / array-like
        The orbital radius of the binary BHs around the lens, in R_Sch.
    source_position: float / array-like
        The source position angle, in units of r_orbit.
        It should be in the range [-1, 1].
    luminosity_distance: float / array-like
        The unperturbed luminosity distance to the source, in Gpc.
    angular_distances: bool, optional
        When True, convert to angular distances when computing theta_E;
        otherwise, luminosity distances are used instead.
        Default is False.

    Returns
    -------
    theta_E: float / array-like
        The Einstein angle in radian.
    beta: float / array-like
        The source position angle in radian.
    lens_mass_source: float / array-like
        The lens mass in the source frame, in solar masses.
    '''
    d_LS = r_orbit * np.sqrt(1 - source_position**2)  # R_Sch
    _source_position = source_position * r_orbit  # R_Sch

    # Schwarschild radius
    # A small, but necessary assumption, that the lens is at dL
    z = np.interp(np.asarray(luminosity_distance, dtype=np.float64), dLGridGlob, zGridGlob)
    lens_mass_source = redshifted_lens_mass / (1 + z)
    R_Sch = 2 * lens_mass_source * MRSUN_SI  # m
    delta = R_Sch / uGpc

    if angular_distances:
        # For most practical purposes, ang_lum_dist = ang_D_S
        ang_lum_dist = luminosity_distance / (1 + z) ** 2  # Gpc
        beta = np.arcsin(_source_position / ang_lum_dist * delta)  # Radian
        ang_D_S = ang_lum_dist * np.cos(beta)   # Gpc
    else:
        beta = _source_position / luminosity_distance * delta  # Radian
        ang_D_S = luminosity_distance

    ang_D_L = ang_D_S / (1 + d_LS * delta)  # Gpc
    theta_E = einstein_angle(lens_mass_source, ang_D_L, d_LS)  # Radian

    return theta_E, beta, lens_mass_source

def einstein_angle(lens_mass_source, angular_D_L, D_LS):
    np = onp
    '''
    Compute the Einstein radius (θ_E) from the given distances.

    We assume D_S = D_L + D_LS, and we assume D_LS is sufficiently
    small such that its (1 + z)^2 correction is unnecessary.

    Making use of the distance hierarchsies, we write:
        θ_E^2 = 2 * (R_S / D_L) * [d_ls / (1 + d_ls)]
        where d_ls = D_LS / D_L

    Parameters
    ----------
    lens_mass_source: float / array-like
        The mass of the lens in the source frame, in solar masses.
    angular_D_L: float / array-like
        The angular distance between the lens and the observer, in Gpc.
    D_LS: float / array-like
        The luminosity distance between the lens and the source, in R_Sch.

    Returns
    -------
    theta_E: float / array-like
        The Einstein radius in radian.
    '''
    RSch = 2 * lens_mass_source * MRSUN_SI  # m
    RSch_2_Gpc = RSch / uGpc

    RSch_DL = RSch_2_Gpc / angular_D_L
    d_ls = D_LS * RSch_DL

    return np.sqrt(2 * RSch_DL * d_ls / (1 + d_ls))


def get_phi_L(iota, y_src_pos, phi_N):
    np = onp
    '''
    Computes the phi_L from the given observer and source positions,
        such that lensing could happen.

    This implies that:
        |δφ| = |φN - φL| < 90º

    In order to recover the sign of δφ, we allow input of
    negative y_src_pos to indicate that.

    Parameters
    ----------
    iota: float / array-like
        Inclination of the observer w.r.t. to the source, radian.
    phi_N: float / array-like
        Azimuthal angle of the observer w.r.t. to the source, radian.
    y_src_pos: float / array-like
        The source position, from (-1, +1), units of r_orbit.
    r_orbit: float / array-like
        The orbital radius of the source around the lens, R_Sch.

    Returns
    -------
    phi_L: float / array-like
        The azimuthal angle of the lens around the source,
        such that lensing could occur.
    '''
    arg = np.sqrt(1 - y_src_pos**2) / np.sin(iota)
    return phi_N - np.sign(y_src_pos) * np.arccos(arg)


def Keplerian_speed(r_orbit):
    '''
    From arXiv:2310.16025, Eq.(2)
        v = (2r - 1)^(-1/2)

    Parameters:
    ----------
    r_orbit: float / array-like
        The orbital radius of the source around the SMBH, R_Sch.

    Returns:
    ----------
    float / array-like
        The orbital speed from Kepler's law, light speed.
    '''

    # TODO: Check whether this is true
    return (2 * r_orbit - 1)**(-0.5)


def gravitational_redshift(r_orbit):
    '''
    From arXiv:2310.16025, Eq.(3)
        z_grav = (1 - 1/r_orbit)^1/2 - 1

    Parameters:
    ----------
    r_orbit: float / array-like
        The orbital radius of the source around the SMBH, R_Sch.

    Returns:
    ----------
    float / array-like
        The gravitational redshift from a Schwarzschild BH.
    '''

    # TODO: Check whether this is true
    return (1 - 1 / r_orbit)**0.5 - 1


def Lorentz_factor(beta):
    return (1 - beta**2)**(-0.5)


def PML_image_position(beta_src, theta_E=1):
    np = onp
    _beta = beta_src / theta_E
    sqrt_term = np.sqrt(4 + _beta**2)
    img_p = (_beta + sqrt_term) / 2 * theta_E
    img_m = (_beta - sqrt_term) / 2 * theta_E
    return img_p, img_m


def PML_time_delay_magnification(beta_src, theta_E=1):
    np = onp
    '''
    Time delay is simplified from https://inspirehep.net/literature/1862768,
    Eq. (3.13), with θ_± = (β ± √ (β² + 4)) / 2, for β measured in units of θ_E.

    t(+) - t(-) = -β √(4 +  β²) + 2 ln(θ(-) / θ(+))

    The magnifications are taken from Eq. (3.8) directly.

    Parameters:
    ----------
    beta_src: float / array-like
        The source position angle.
        If theta_E is not given, it is assumed to be in unit of theta_E.
        Otherwise, it should have the same unit as theta_E (radian, R_Sch, etc).
    theta_E: float / array-like
        The Einstein radius (angle), it acts as the scale of for beta_src.
        It can have any units, as long as it being consistent with beta_src.

    Returns:
    ----------
    delta_t: float / array-like
        The time delay between the "+" and "-" images, in geometric time.
        (Need to multiply by GM/c^3 to get SI unit.)
    mag_p, mag_m: float / array-like
        The magnification of the "+" and "-" images respectively.
    '''

    img_p, img_m = PML_image_position(beta_src, theta_E)

    _beta = beta_src / theta_E
    sqrt_term = np.sqrt(4 + _beta**2)
    delta_t_geom = - _beta * sqrt_term
    delta_t_Shap = 2 * np.log(np.abs(img_m / img_p))

    # The factor of 2 is to account for the later multiplication by GM/c^3
    delta_t = (delta_t_geom + delta_t_Shap) * 2

    common_term = _beta / sqrt_term + sqrt_term / _beta
    mag_p = 0.25 * (common_term + 2)
    mag_m = 0.25 * (common_term - 2)

    return delta_t, mag_p, mag_m


## Test random covariance matrix

In [114]:
# a random covariance matrix
seed = 13
onp.random.seed(seed)

dim = 14
matrices = onp.random.uniform(size=(dim, dim, 2)).astype(onp.float128)
sym_mat = 0.5 * (matrices + matrices.transpose((1, 0, 2))) + dim * onp.eye(dim)[:, :, None]


In [ ]:
reference_parameters = {
    'Mc': 30, 'eta': 0.24, 'iota': 0.9*onp.pi/2, 'phase': 0.9*onp.pi/2,
    'chi1z': 0.3, 'chi2z': 0.5, 'tcoal': 0,
    'R_orbit': 100, 'M_lz': 1e4, 'src_pos': 0.5,
    'dL': 1, 'psi': 4, 'theta': 1.87, 'phi': 2.66,
}

from_params = ['iota', 'R_orbit', 'src_pos', 'M_lz']

lensing_parameters = {key: onp.full(2, val).astype(onp.float128) for key, val in reference_parameters.items()}

transformed_cov_mat, transformed_parameters, transformed_keys = covariance_change_variable(
        sym_mat[:, :, 1], lensing_parameters, lensing_transform, from_params
    )

print("== Condition number (Original) ==", mp_cond(sym_mat[:, :, 1].astype(onp.float128)))

if onp.all(onp.isfinite(transformed_cov_mat)):
    print("== Condition number (Transformed) ==", mp_cond(transformed_cov_mat))
else:
    print("Warning: transformed covariance matrix contains non-finite values (nan/inf), skipping diagnostics.")
    print("  nan count:", onp.sum(~onp.isfinite(transformed_cov_mat)))


== Condition number (Jacobian sub-block) == 16271632.383328812
== Diagonal elements (Jacobian) ==
  Mc: 1.0
  eta: 1.0
  delta_iota: -0.5021873195065751
  phase: 1.0
  chi1z: 1.0
  chi2z: 1.0
  tcoal: 1.0
  delta_phi: -0.0013766765505351941
  delta_time: -0.00017985612998927536
  relative_mass: -0.0064648286723922865
  dL: 1.0
  psi: 1.0
  theta: 1.0
  phi: 1.0
== Maximum absolute value (Jacobian) == 6.260246765421584086
== Minimum absolute value (Jacobian) == 0.0
== log|det(C)| (original covariance) == 1 37.29679585609441
== log|det(J)| (sub-Jacobian, = det of full J) == 1 -20.125477040361535
== Expected log|det(J C J^T)| = 2*logdetJ + logdetC == 1 -2.9541582246286566
== log|det(J C J^T)| (transformed covariance) == 1 -2.9541582168496534
== Condition number (Original) == 1.6493130859101237
== Condition number (Transformed) == 258707782804639.9


In [118]:
lensing_transform(reference_parameters)

{'delta_iota': np.float64(0.07759221239828129),
 'delta_phi': np.float64(0.275382043203642),
 'relative_mass': np.float64(1.0172136750750245),
 'delta_time': np.float64(-1.8013092835690026)}

## Test GW covariance matrix

In [122]:
# Set up detectors
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

wf_model = waveforms.IMRPhenomD()

H1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=H1, fmin=10)
L1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=L1, fmin=10)
V1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=V1, fmin=10)
HLV_AGN = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1


In [137]:
shape = 3
reference_parameters = {
    'Mc': 80, 'eta': 0.23, 'iota': 0.99 * np.pi / 2, 'phase': 2.1,
    'chi1z': 0.3, 'chi2z': 0.4, 'tcoal': 0.0,
    'theta': 0.1, 'phi': 0.3, 'psi': 0.3, 'dL': 2.0, 
    'R_orbit': 100, 'M_lz': 5e5, 'src_pos': 0.2
}
lensing_parameters = {key: np.full(shape, val).astype(np.float64) for key, val in reference_parameters.items()}
# lensing_parameters['src_pos'] = np.arange(1, 6, 2) * 0.05
lensing_parameters['R_orbit'] = np.arange(50., 200., 70.)

HLV_fisher = HLV_AGN.FisherMatr(lensing_parameters, res=1000)

keys = list(reference_parameters.keys()).copy()
reduce_fisher_mat, _ = reduce_Fisher_matrix(HLV_fisher, keys=keys)
# Convert to regular numpy for float128 support
reduce_fisher_mat = onp.array(reduce_fisher_mat)
reduced_cov_mats, ie = compute_covariance_matrix(reduce_fisher_mat)

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.


In [ ]:
reduced_cov_mats.shape

(14, 14, 3)

In [154]:
from_params = ['iota', 'R_orbit', 'src_pos', 'M_lz']

lensing_parameters = {key: onp.full(2, val).astype(onp.float128) for key, val in reference_parameters.items()}

transformed_cov_mat, transformed_parameters, transformed_keys = covariance_change_variable(
        reduced_cov_mats[:, :, 2], lensing_parameters, lensing_transform, from_params
    )

print("== Condition number (Original) ==", mp_cond(reduced_cov_mats[:, :, 0].astype(onp.float128)))

if onp.all(onp.isfinite(transformed_cov_mat)):
    print("== Condition number (Transformed) ==", mp_cond(transformed_cov_mat))
else:
    print("Warning: transformed covariance matrix contains non-finite values (nan/inf), skipping diagnostics.")
    print("  nan count:", onp.sum(~onp.isfinite(transformed_cov_mat)))

== Condition number (Jacobian sub-block) == 137221748.19165128

== Full Jacobian (rows=output params, cols=input params) ==
                             Mc         eta        iota       phase       chi1z       chi2z       tcoal       theta         phi         psi          dL     R_orbit        M_lz     src_pos
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
                 Mc           1           0           0           0           0           0           0           0           0           0           0           0           0           0
                eta           0           1           0           0           0           0           0           0           0           0           0           0           0           0
         delta_iota           0           0       -1.39           0           0           0           0           0         

In [155]:
lensing_transform(reference_parameters)

{'delta_iota': np.float64(0.021767382695727644),
 'relative_mass': np.float64(1.020064476539571),
 'relative_distance': np.float64(3.929506928831463),
 'delta_time': np.float64(-25.368068279871192)}

In [156]:
_print_matrix_table(reduced_cov_mats[:, :, 2], keys, keys)

                   Mc         eta        iota       phase       chi1z       chi2z       tcoal       theta         phi         psi          dL     R_orbit        M_lz     src_pos
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
       Mc        11.6      0.0465    -0.00547         1.4      -0.942        2.16      0.0519    0.000504      0.0208      0.0166       0.555        -297    3.11e+04       0.153
      eta      0.0465     0.00104    8.21e-05       0.105     -0.0371      0.0707      0.0019    -1.1e-06    -0.00029   -0.000335     0.00421      -0.648         239    0.000334
     iota    -0.00547    8.21e-05      0.0086      0.0188    -0.00278     0.00641    0.000195   -0.000231     -0.0301     -0.0292   -0.000614        3.06        -390    -0.00146
    phase         1.4       0.105      0.0188        12.3       -4.11         7.7       0.209    -0.00044     